# AASTU Student Handbook Question-Answering Assistant
### Complete End-to-End Grounded RAG Project (Google Colab Ready)
**AWS AI/ML AASTU Cohort · Week 4 Assignment**

This notebook is **100% self-contained and independent**. You can run it directly in **Google Colab** from top to bottom. It contains all phases of the project:
1. **Automated Data Setup**: Downloads or loads the official 375-page AASTU Student Handbook PDF.
2. **Phase 1: Ingestion & Continuous Chunking**: Extracts text from all pages and applies continuous document-level chunking with a 50-word cross-page overlap.
3. **Phase 2: Dense Vector Store (ChromaDB)**: Embeds and indexes chunks into ChromaDB with cosine similarity.
4. **Phase 3: Prompt Engineering & Gemini Engine**: Implements the system prompt, Pydantic structured output validation, and a bounded repair loop.
5. **Phase 4: Guardrails & Safe Refusal**: Demonstrates out-of-scope hallucination prevention.
6. **Phase 5: Golden Regression Evaluation Suite**: Evaluates 8 test cases and outputs benchmark tables.
7. **Phase 6: Interactive Q&A Console**: Allows you to query any academic policy in real time.


## 1. Install Dependencies
Run the cell below to install all required libraries in your Colab or local environment.


In [ ]:
%pip install -q pypdf chromadb requests pydantic python-dotenv


## 2. Imports & API Key Configuration
Configure your Google Gemini API key. If running in Google Colab, you can enter it interactively or use Colab Secrets.


In [ ]:
import os
import sys
import re
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
import requests
from pydantic import BaseModel, Field, ValidationError

# Get Gemini API key
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("APIKEY")
if not api_key:
    # Try loading from local .env if present
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("APIKEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Google Gemini API Key: ").strip()

print(f" Gemini API Key configured: {api_key[:6]}...{api_key[-4:]}")


## 3. Data Acquisition: AASTU Student Handbook PDF
The cell below automatically checks for the handbook. If running on Google Colab or on a new machine, it downloads the official handbook directly from the GitHub repository.


In [ ]:
PDF_FILENAME = "aastu_handbook.pdf"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True, parents=True)

pdf_candidates = [
    Path(PDF_FILENAME),
    DATA_DIR / PDF_FILENAME,
    Path("week-04/data/aastu_handbook.pdf"),
    Path("../week-04/data/aastu_handbook.pdf"),
    Path("d:/projectFolder/AWS/week1-demo/week-04/data/aastu_handbook.pdf")
]

pdf_path = None
for cand in pdf_candidates:
    if cand.exists():
        pdf_path = cand
        break

if pdf_path is None or not pdf_path.exists():
    print("[*] Handbook PDF not found locally. Downloading from GitHub...")
    url = "https://raw.githubusercontent.com/AbellTT/aws-sbg-tasks/main/week-04/data/aastu_handbook.pdf"
    pdf_path = DATA_DIR / PDF_FILENAME
    resp = requests.get(url, stream=True)
    if resp.status_code == 200:
        with open(pdf_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f" Downloaded {pdf_path.name} ({pdf_path.stat().st_size / (1024*1024):.2f} MB)")
    else:
        print("[!] Direct download failed. Please upload 'aastu_handbook.pdf' via the file browser.")
        try:
            from google.colab import files
            uploaded = files.upload()
            pdf_path = Path(list(uploaded.keys())[0])
        except ImportError:
            raise FileNotFoundError("Please provide aastu_handbook.pdf")

print(f" Active PDF Path: {pdf_path.resolve()} ({pdf_path.stat().st_size / (1024*1024):.2f} MB)")


## 4. Phase 1: PDF Extraction & Continuous Document Chunking

### Why Continuous Chunking?
Traditional chunking splits each page independently. That creates a critical blind spot: sentences and policies split across page boundaries (e.g. Page 25 to Page 26) lose their context.

Here, we implement **continuous document-level chunking**:
1. Extract all text into a unified word sequence while tracking exact page boundaries.
2. Roll a **350-word window** with a **50-word sliding overlap** across the entire document.
3. Dynamically map every chunk back to its source pages (`citation_label`).


In [ ]:
from pypdf import PdfReader

def clean_text(text: str) -> str:
    """Cleans extracted PDF text while preserving sentence boundaries."""
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def load_and_extract_pdf(path: Path) -> List[Dict[str, Any]]:
    """Extracts text page by page from the AASTU Student Handbook PDF."""
    reader = PdfReader(str(path))
    pages_data = []
    total_pages = len(reader.pages)
    print(f"[*] Extracting text from {total_pages} pages...")

    for idx, page in enumerate(reader.pages, 1):
        raw_text = page.extract_text() or ""
        cleaned = clean_text(raw_text)
        if cleaned:
            pages_data.append({"page": idx, "text": cleaned})

    print(f" Extracted {len(pages_data)} non-empty pages from {total_pages} total pages.")
    return pages_data

def create_continuous_chunks(
    pages_data: List[Dict[str, Any]],
    chunk_size: int = 350,
    chunk_overlap: int = 50
) -> List[Dict[str, Any]]:
    """
    Builds continuous document-level chunks with cross-page overlap and accurate page citation labels.
    """
    full_words = []
    word_page_map = []

    for item in pages_data:
        p_num = item["page"]
        p_words = item["text"].split()
        for w in p_words:
            full_words.append(w)
            word_page_map.append(p_num)

    total_words = len(full_words)
    print(f"[*] Total handbook document words: {total_words}")

    chunks = []
    step = chunk_size - chunk_overlap
    chunk_idx = 0

    for start_idx in range(0, total_words, step):
        end_idx = min(start_idx + chunk_size, total_words)
        chunk_words = full_words[start_idx:end_idx]
        chunk_pages = sorted(list(set(word_page_map[start_idx:end_idx])))

        page_start = chunk_pages[0]
        page_end = chunk_pages[-1]
        citation = f"Page {page_start}" if page_start == page_end else f"Pages {page_start}–{page_end}"

        chunks.append({
            "chunk_id": f"chunk_{chunk_idx:04d}",
            "text": " ".join(chunk_words),
            "word_count": len(chunk_words),
            "page_start": page_start,
            "page_end": page_end,
            "pages": chunk_pages,
            "citation_label": citation,
            "source": "AASTU Student Handbook"
        })
        chunk_idx += 1

        if end_idx >= total_words:
            break

    print(f" Generated {len(chunks)} continuous chunks with {chunk_overlap}-word cross-page overlap.")
    return chunks

# Execute extraction and chunking
pages_data = load_and_extract_pdf(pdf_path)
chunks = create_continuous_chunks(pages_data, chunk_size=350, chunk_overlap=50)

# Inspect a chunk showing cross-page boundary
cross_page_chunk = next((c for c in chunks if c["page_start"] != c["page_end"]), chunks[0])
print(f"\nSample Cross-Page Chunk: ID={cross_page_chunk['chunk_id']}, Citation='{cross_page_chunk['citation_label']}'")
print(f"Preview: {cross_page_chunk['text'][:180]}...")


## 5. Phase 2: ChromaDB Dense Semantic Vector Store
We index the continuous chunks into a local ChromaDB collection using `all-MiniLM-L6-v2` with cosine distance space (`hnsw:space: cosine`).


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize in-memory or persistent ChromaDB
chroma_client = chromadb.Client()
COLLECTION_NAME = "aastu_handbook_colab"

# Reset collection if re-running
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

embedding_fn = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Batch index chunks
batch_size = 64
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{
            "citation_label": c["citation_label"],
            "page_start": c["page_start"],
            "page_end": c["page_end"],
            "word_count": c["word_count"],
            "source": c["source"]
        } for c in batch]
    )

print(f" Successfully indexed {collection.count()} chunks into ChromaDB!")

def retrieve_context(query: str, top_k: int = 3) -> List[Dict[str, Any]]:
    """Retrieves top_k most semantically relevant chunks for a student query."""
    results = collection.query(query_texts=[query], n_results=top_k)
    hits = []
    for idx in range(len(results["ids"][0])):
        hits.append({
            "chunk_id": results["ids"][0][idx],
            "text": results["documents"][0][idx],
            "distance": round(results["distances"][0][idx], 4),
            "citation_label": results["metadatas"][0][idx]["citation_label"],
            "page_start": results["metadatas"][0][idx]["page_start"],
            "page_end": results["metadatas"][0][idx]["page_end"]
        })
    return hits

# Test retrieval
sample_hits = retrieve_context("What are the criteria for academic dismissal or probation?", top_k=2)
for i, h in enumerate(sample_hits, 1):
    print(f"Hit {i}: {h['citation_label']} (Distance: {h['distance']:.3f}) | {h['text'][:100]}...")


## 6. Phase 3: Prompt Engineering & Structured Output Validation

We define:
1. **GroundedAnswer Pydantic Schema**: Guarantees answer, citations list, confidence score, and boolean `can_answer` refusal flag.
2. **System Prompt Template**: Enforces strict grounding, anti-hallucination rules, and page citation format.
3. **Resilient REST Engine**: Bypasses SDK connection issues and features an exponential backoff loop for rate limits (HTTP 429/503).
4. **Bounded Repair Loop**: Corrects any malformed LLM JSON output automatically.


In [ ]:
class GroundedAnswer(BaseModel):
    answer: str = Field(description="Clear, grounded response answering the student inquiry.")
    citations: List[str] = Field(default_factory=list, description="List of exact page labels cited, e.g. ['Pages 25–27'].")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score between 0.0 and 1.0.")
    can_answer: bool = Field(description="True if context answers the question; False if out-of-scope.")

SYSTEM_PROMPT_TEMPLATE = """You are the official Addis Ababa Science and Technology University (AASTU) Student Handbook AI Assistant.
Your mission is to provide accurate, strictly grounded answers to student inquiries based exclusively on the provided handbook excerpts.

==================================================
OPERATIONAL INSTRUCTIONS & ANTI-HALLUCINATION RULES:
==================================================
1. STRICT GROUNDING: Answer SOLELY using facts explicitly stated in the CONTEXT below. Do NOT extrapolate or assume.
2. CITATION ENFORCEMENT: Every claim must be cited with its exact page range, e.g. "Pages 25–27".
3. SAFE REFUSAL: If the context does not contain sufficient information, you MUST set "can_answer": false, "citations": [], and clearly state that the information is not in the handbook.
4. JSON OUTPUT CONTRACT: Output ONLY a JSON object adhering to the schema:
{
  "answer": "string",
  "citations": ["Pages X-Y"],
  "confidence": 0.0,
  "can_answer": true
}

==================================================
CONTEXT EXCERPTS FROM AASTU STUDENT HANDBOOK:
==================================================
{context}

==================================================
STUDENT QUESTION:
==================================================
{question}
"""

def call_gemini_raw(prompt: str, api_key: str, model: str = "gemini-3.1-flash-lite", timeout: int = 40) -> str:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={api_key}"
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseMimeType": "application/json", "temperature": 0.1}
    }
    for attempt in range(3):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            if resp.status_code in (429, 503):
                time.sleep((attempt + 1) * 3)
                continue
            resp.raise_for_status()
            data = resp.json()
            return data["candidates"][0]["content"]["parts"][0]["text"]
        except Exception as e:
            if attempt == 2:
                # Try fallback
                fallback_url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent?key={api_key}"
                resp = requests.post(fallback_url, json=payload, timeout=timeout)
                resp.raise_for_status()
                return resp.json()["candidates"][0]["content"]["parts"][0]["text"]
            time.sleep(3)
    raise RuntimeError("Failed to query Gemini API.")

def generate_grounded_answer(prompt: str, api_key: str, model: str = "gemini-3.1-flash-lite") -> Tuple[GroundedAnswer, int]:
    current_prompt = prompt
    for attempt in range(1, 4):
        raw_text = call_gemini_raw(current_prompt, api_key, model=model)
        cleaned = raw_text.strip()
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
            cleaned = re.sub(r"\s*```$", "", cleaned)
        try:
            return GroundedAnswer.model_validate_json(cleaned), attempt
        except Exception as err:
            if attempt == 3:
                raise
            current_prompt += f"\n\n[REPAIR DIRECTIVE] Your JSON was invalid: {err}. Output strictly valid JSON."

def ask_handbook(query: str, top_k: int = 3, api_key: str = api_key) -> Dict[str, Any]:
    hits = retrieve_context(query, top_k=top_k)
    context_str = "\n\n".join([f"--- Context Chunk [{h['citation_label']}] ---\n{h['text']}" for h in hits])
    rendered_prompt = SYSTEM_PROMPT_TEMPLATE.replace("{context}", context_str).replace("{question}", query)
    ans, attempts = generate_grounded_answer(rendered_prompt, api_key)
    return {
        "query": query,
        "answer": ans.answer,
        "citations": ans.citations,
        "confidence": ans.confidence,
        "can_answer": ans.can_answer,
        "retrieved_hits": hits,
        "attempts": attempts
    }

print(" Grounded RAG Engine ready!")


## 7. Grounded Response Demonstration (In-Scope Query)
Testing an academic query covered directly by the handbook.


In [ ]:
test_query = "What is the policy and penalty for late course registration?"
result = ask_handbook(test_query, top_k=3)

print("=" * 75)
print(f"QUERY:       {result['query']}")
print(f"ANSWER:      {result['answer']}")
print(f"CITATIONS:   {result['citations']}")
print(f"CONFIDENCE:  {result['confidence']}")
print(f"CAN ANSWER:  {result['can_answer']} (Grounded in context)")
print("=" * 75)


## 8. Anti-Hallucination & Adversarial Safe Refusal
Testing an out-of-scope query. The model must refuse safely with `can_answer: false` and empty citations rather than hallucinating.


In [ ]:
adv_query = "How do I bake a chocolate cake at home?"
adv_result = ask_handbook(adv_query, top_k=2)

print("=" * 75)
print(f"QUERY:       {adv_result['query']}")
print(f"ANSWER:      {adv_result['answer']}")
print(f"CITATIONS:   {adv_result['citations']} (Expected: Empty)")
print(f"CAN ANSWER:  {adv_result['can_answer']} (Expected: False - Safe Refusal)")
print("=" * 75)


## 9. Phase 4: Golden Regression Evaluation Benchmark
We run 8 regression test cases: 6 covering distinct handbook policies (dismissal, late registration, make-up exams, student records, warning policy, transfer credits) and 2 adversarial out-of-scope queries.


In [ ]:
golden_dataset = [
    {
        "id": "tc_01",
        "category": "Academic Policy",
        "query": "What are the criteria for academic dismissal and how can a student be placed on probation?",
        "is_in_scope": True,
        "expected_keywords": ["dismissal", "probation", "SGPA", "Academic Commission"],
        "expected_page_range": [25, 28]
    },
    {
        "id": "tc_02",
        "category": "Course Registration",
        "query": "What is the procedure and policy for late registration of courses?",
        "is_in_scope": True,
        "expected_keywords": ["registration", "registrar", "late"],
        "expected_page_range": [17, 19]
    },
    {
        "id": "tc_03",
        "category": "Examinations",
        "query": "Under what circumstances can a student be allowed to take a make-up examination?",
        "is_in_scope": True,
        "expected_keywords": ["make-up", "examination", "GPA"],
        "expected_page_range": [28, 30]
    },
    {
        "id": "tc_04",
        "category": "Student Records",
        "query": "Can student academic records and transcripts be disclosed to third parties without consent?",
        "is_in_scope": True,
        "expected_keywords": ["consent", "third party", "transcripts"],
        "expected_page_range": [32, 34]
    },
    {
        "id": "tc_05",
        "category": "Academic Status",
        "query": "What happens if a student receives consecutive academic warnings?",
        "is_in_scope": True,
        "expected_keywords": ["warning", "dismissal", "probation"],
        "expected_page_range": [24, 28]
    },
    {
        "id": "tc_06",
        "category": "Transfer & Credits",
        "query": "How are previously attended courses examined when transferring credits between departments?",
        "is_in_scope": True,
        "expected_keywords": ["department", "course", "Registrar"],
        "expected_page_range": [31, 33]
    },
    {
        "id": "tc_07",
        "category": "Adversarial (Out of Scope)",
        "query": "How do I bake a chocolate cake at home?",
        "is_in_scope": False,
        "expected_keywords": [],
        "expected_page_range": []
    },
    {
        "id": "tc_08",
        "category": "Adversarial (Out of Scope)",
        "query": "What is the capital city of Australia and what is its population?",
        "is_in_scope": False,
        "expected_keywords": [],
        "expected_page_range": []
    }
]

eval_results = []
print("=" * 80)
print("  AASTU STUDENT HANDBOOK: RUNNING GOLDEN EVALUATION SUITE")
print("=" * 80)

for idx, tc in enumerate(golden_dataset, 1):
    q = tc["query"]
    in_scope = tc["is_in_scope"]
    res = ask_handbook(q, top_k=3, api_key=api_key)
    hits = res["retrieved_hits"]
    ans = res["answer"]
    cits = res["citations"]
    can_ans = res["can_answer"]

    # Evaluation checks
    retrieval_hit = False
    if in_scope and tc.get("expected_page_range"):
        min_p, max_p = tc["expected_page_range"]
        for h in hits:
            if any(min_p <= p <= max_p for p in range(h["page_start"], h["page_end"] + 1)):
                retrieval_hit = True
                break
    elif not in_scope:
        retrieval_hit = True

    dec_ok = (can_ans == in_scope)
    cit_ok = (len(cits) > 0) if in_scope else (len(cits) == 0)
    passed = dec_ok and cit_ok and retrieval_hit

    status = "PASS [OK]" if passed else "FAIL [X]"
    print(f"[{idx}/8] {status} | {tc['category']}: "{q[:45]}..."")
    eval_results.append({
        "id": tc["id"],
        "category": tc["category"],
        "query": q,
        "hit": retrieval_hit,
        "decision": "Answered" if can_ans else "Refused (Safe)",
        "citations": ", ".join(cits) if cits else "None (Safe)",
        "passed": passed
    })
    time.sleep(1.5)

passed_count = sum(1 for r in eval_results if r["passed"])
pass_rate = (passed_count / len(eval_results)) * 100
print("\n" + "=" * 80)
print(f"  BENCHMARK SUMMARY: {pass_rate:.1f}% ({passed_count}/{len(eval_results)} Tests Passed)")
print("=" * 80)


## 10. Benchmark Results Table


In [ ]:
import pandas as pd
df = pd.DataFrame(eval_results)
df.columns = ["Test ID", "Category", "Query", "Retrieval Hit", "Decision", "Citations", "Passed"]
df


## 11. Interactive Student Handbook Assistant
Type any question below to test the assistant interactively.


In [ ]:
user_question = input("Ask a question about AASTU handbook (or press enter for default): ").strip()
if not user_question:
    user_question = "Under what circumstances can a student be allowed to take a make-up examination?"

print(f"\nQuery: {user_question}")
response = ask_handbook(user_question, top_k=3)
print("\n" + "=" * 70)
print("ANSWER:")
print(response["answer"])
print("-" * 70)
print(f"Citations:  {response['citations']}")
print(f"Confidence: {response['confidence']:.2f}")
print("=" * 70)
